In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import multivariate_normal


**Her indlæses inputdata fra Excel-filerne.**  
Tilpas filnavne og eventuelle kolonne-/arknavne efter dine behov.


In [ ]:
# Load initial values and covariance matrix
init_values = pd.read_excel("init_values.xlsx", index_col=0)  # HER kan du ændre filnavn
covariance_matrix = pd.read_excel("covariance_matrix.xlsx", index_col=0)  # HER kan du ændre filnavn

# Vis data for verificering
print("Initial Values:")
print(init_values)
print("\nCovariance Matrix:")
print(covariance_matrix)


**Parametre for simulationen**  
Vi antager en uges tidssteg, en investeringshorisont på 1 år, og et antal simuleringer.  
Du kan justere antallet af simuleringer og tidssteg efter behov.


In [ ]:
time_steps = 52  # Weekly steps (1 year horizon)
num_simulations = 10000
horizon = 1
dt = horizon / time_steps


**Definer mean-vektoren (\(\mu\)) i henhold til opgaven**  
Ifølge opgaveteksten:  
\[
\mu = (0, 0.07 \cdot \Delta t, 0.06 \cdot \Delta t, 0^\top, 0^\top)^\top
\]

Du skal identificere de korrekte indeks i din \(\Delta X\)-vektor. Antag fx følgende rækkefølge i \(\Delta X\)-vektoren:
- index_fx = indeks for \(\Delta \log FX_t\)
- index_VUS = indeks for \(\Delta \log V_{US,local}\)
- index_VEUR = indeks for \(\Delta \log V_{EUR}\)
- De resterende elementer er renteændringer og får 0 i mean.

**NB:** Du skal selv tilpasse disse indeks til din opsætning.


In [ ]:
# HER skal du selv indstille indeks efter din datastruktur
# Eksempel (disse skal rettes til!):
index_fx = 0
index_VUS = 1
index_VEUR = 2
# De resterende er renter. Antal dimensioner i covariance_matrix:
dim = covariance_matrix.shape[0]

mu = np.zeros(dim)
mu[index_VUS] = 0.07 * dt
mu[index_VEUR] = 0.06 * dt

sigma = covariance_matrix.values
initial_values_array = init_values['initial values'].values


**Simulér udviklingen i markedsinvarianter \(\Delta X\)**  
Vi trækker tilfældige multivariate normalfordelte increments med middelværdi \(\mu\) og kovarians \(\Sigma \cdot \Delta t\).


In [ ]:
increments = np.random.multivariate_normal(mu, sigma * dt, size=(num_simulations, time_steps))


**Konverter log-inkrementer til niveauer**  
For FX og aktier simuleres \(\Delta \log\)-værdier. Ved periodens slutning skal vi eksponentiere sum af inkrementer plus initialt logniveau.  
For renter (yields) antager vi additive inkrementer på yield-niveauerne.

Her antages \(\Delta X\) ordnet som:  
- fx_spot = log-niveau  
- V_US_local = log-niveau af US equity (local)  
- V_EUR = log-niveau af EUR equity  
- De resterende elementer er yields i forskellige løbetider for EUR og USD.

**Du skal selv sikre, at rækkefølgen af yields passer med dine data.**  
I dette eksempel antages en bestemt rækkefølge (tilpas denne del til din datastruktur).
